# Phase 6 - Validate the way the client validates, then stress it

Reproduce the client's diagnostic (Pearson r between July annual means, n = years) for
both models on the same lakes and field data, then add the three things the client did
not: a bootstrap CI on r, a sensitivity grid over the analyst's discretionary choices,
and a provenance check of Water Quality Portal Secchi against LAGOS-US LIMNO.

The credibility anchor: the national model should fail on our Adirondack lakes too.
Reproducing the client's failure on different lakes, before claiming to fix it, proves
the failure is structural rather than specific to Squam.

In [ ]:
# Cell 1 of 5: repo + Drive
import os, pathlib, subprocess, sys

REPO = "https://github.com/cy6erlizard/landsat-lake-clarity-adirondack.git"
ROOT = pathlib.Path("/content/landsat-lake-clarity-adirondack")
if ROOT.exists():
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(ROOT)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)], check=True)

from google.colab import drive
drive.mount("/content/drive")
os.environ["LAKECLARITY_DATA"] = "/content/drive/MyDrive/lake-clarity"

In [ ]:
# Cell 2 of 5: pull field Secchi from the Water Quality Portal REST API
import pandas as pd
from lakeclarity import config, locus, validate, viz, wqp

viz.use_style()
targets = pd.read_csv(config.TABLE_DIR / "T05_target_lakes.csv", index_col=0).squeeze()
target_ids = [int(targets["large"]), int(targets["small"])]
lakes = locus.load_lakes().set_index("lagoslakeid")

# Confirm the characteristic-name choice before relying on it.
coverage = wqp.characteristic_coverage()
print(coverage.to_string(index=False))

field = wqp.fetch_secchi()  # cached to Drive after the first pull
print(f"\n{len(field):,} Secchi records in the region, {field.year.min()}-{field.year.max()}")

In [ ]:
# Cell 3 of 5: the like-for-like comparison, with bootstrap intervals
predictions = pd.read_parquet(config.PROCESSED_DIR / "predictions_full.parquet")
national = pd.read_parquet(config.INTERIM_DIR / "national_predictions_region.parquet")
national = national.rename(columns={"Secchi_predicted": "secchi_predicted_m"})
national["sensing_dt"] = pd.to_datetime(national["SENSING_TIME"], format="ISO8601", utc=True)
national["year"] = national["sensing_dt"].dt.year
national["month"] = national["sensing_dt"].dt.month

# Field Secchi per target lake: match WQP site to lake by nearest location, or
# supply the client's own station list here when provided.
results = []
for lake_id in target_ids:
    name = lakes.loc[lake_id, "lake_name"]
    obs = field  # replace with the lake's stations once known
    for label, pred in [("regional", predictions), ("national", national)]:
        res = validate.july_validation(pred, obs, lake_id, name, label)
        if res:
            results.append(res)
            print(res.sentence())

In [ ]:
# Cell 4 of 5: does the result survive the analyst's choices?
# Rebuild predictions under each configuration, then grid the headline r.
from itertools import product

configs = {}
for window, floor, qa, months in product([1, 3], [10, 25], ["on"], [(7,), (6, 7, 8)]):
    p = predictions[predictions["Pixelcount"] >= floor]
    configs[(window, floor, qa, months)] = p

for lake_id in target_ids:
    grid = validate.sensitivity_grid(configs, field, lake_id)
    grid.to_csv(config.TABLE_DIR / f"T13_sensitivity_{lake_id}.csv", index=False)
    print(f"\nlake {lake_id} ({lakes.loc[lake_id, 'lake_name']}):")
    print(grid.to_string(index=False))
    if grid["r"].notna().any():
        print(f"  r ranges [{grid['r'].min():+.2f}, {grid['r'].max():+.2f}] across the grid")

In [ ]:
# Cell 5 of 5: figures F25 to F28
matchups = pd.read_parquet(config.INTERIM_DIR / "matchups.parquet") if (config.INTERIM_DIR / "matchups.parquet").exists() else None

for lake_id in target_ids:
    name = lakes.loc[lake_id, "lake_name"]
    print("wrote", viz.save(validate.fig_validation_overlay(
        field, predictions, national, lake_id, name), "F25", f"overlay_{lake_id}").name)
    grid = validate.sensitivity_grid(configs, field, lake_id)
    print("wrote", viz.save(validate.fig_sensitivity_grid(grid, name), "F27", f"sensitivity_{lake_id}").name)
    if matchups is not None:
        merged = validate.provenance_check(field, matchups, lake_id)
        if len(merged) > 2:
            print("wrote", viz.save(validate.fig_provenance(merged, name), "F28", f"provenance_{lake_id}").name)

print("wrote", viz.save(validate.fig_bootstrap_r(results), "F26", "bootstrap_r").name)